# BirdNET Embedding EDA
This notebook explores the embeddings generated by the **BirdNET-Analyzer V2.4** model and evaluates their quality for identifying species in the BirdCLEF 2026 dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os

# Load BirdNET results
processed_dir = '../data/processed'
embeddings_path = os.path.join(processed_dir, 'birdnet_embeddings.npz')

if os.path.exists(embeddings_path):
    data = np.load(embeddings_path)
    embeddings = data['embeddings']
    indices = data['indices']
    print(f"Loaded {len(embeddings)} embeddings with shape {embeddings.shape}")
else:
    print("BirdNET embeddings not found. Please run the extraction script first.")

## 1. Dimensionality Reduction (t-SNE)
We use t-SNE to see if BirdNET naturally clusters the species in our dataset.

In [ ]:
if 'embeddings' in locals():
    # Load metadata to get labels
    train_df = pd.read_csv('../data/raw/train.csv')
    labels = train_df.iloc[indices]['primary_label'].values
    
    # Reduce dimensions for visualization
    print("Running t-SNE...")
    # Note: 6522 dimensions is high for t-SNE directly, usually PCA first is better
    pca = PCA(n_components=50)
    X_pca = pca.fit_transform(embeddings)
    
    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto')
    X_tsne = tsne.fit_transform(X_pca)
    
    # Plot
    plt.figure(figsize=(12, 8))
    # Only color the top 10 species for clarity
    top_species = pd.Series(labels).value_counts().head(10).index
    
    for species in top_species:
        mask = labels == species
        plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=species, alpha=0.6, s=10)
    
    plt.title("t-SNE Visualization of BirdNET Embeddings (Top 10 Species)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

## 2. Embedding Statistics
BirdNET produces a 6,522-dimensional logit vector. Let's look at the distribution.

In [ ]:
if 'embeddings' in locals():
    plt.figure(figsize=(10, 4))
    sns.histplot(embeddings.flatten()[:10000], bins=50, kde=True)
    plt.title("Distribution of BirdNET Output Values")
    plt.xlabel("Value")
    plt.show()
    
    print(f"Mean: {embeddings.mean():.4f}")
    print(f"Std Dev: {embeddings.std():.4f}")
    print(f"Min: {embeddings.min():.4f}")
    print(f"Max: {embeddings.max():.4f}")